# PPO + Basic CNN: short steering-stability experiment

Start from the selected **400,000-transition** model. Train a separate copy on ordinary seeds **1000–1099**. Public, validation and final-test seeds are excluded from training. The CNN, PPO and gSDE architecture stay the same.

**Hypothesis:** a small cost for abrupt changes in the executed steering command may reduce oscillation. This is unverified and can also hurt cornering. No faster lap is promised. The original reward already encourages road progress and penalizes elapsed simulator time. We add no gas bonus and no hand-written driving rules.

The experiment uses a lower learning rate (`1e-4`) and a steering-change cost (`0.01`). Because two settings change, this run cannot identify which one caused a result. The training budget is **at most about 10k additional transitions or 12 minutes of collection/training**, whichever stops first. Setup, evaluation and saving take additional time; half an hour is a target, not a guarantee. If setup/evaluation is already slow, keep the released baseline.

This notebook is self-contained. Upload to Colab, select an available GPU, and use the same runtime/device for both comparisons. Keep your published baseline unchanged. This is a local experiment notebook; it does not publish a release.

## 1. Install and import
Run these cells first. A runtime restart clears variables: rerun setup after restarting.

In [ ]:
%pip install -q "swig==4.*"
%pip install -q "gymnasium[box2d]==1.3.0" "stable-baselines3==2.9.0" "torch>=2.8,<3.0" "numpy>=1.26,<3" "pillow>=10,<13" "matplotlib>=3.8,<4" "tensorboard>=2.16,<3" "imageio[ffmpeg]>=2.34,<3" "tqdm>=4.66,<5" "rich>=13"


In [ ]:
import os
# Headless SDL: Colab has no desktop window. Images/video are displayed inline.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import json
import sys
import time
import importlib.metadata as metadata
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import imageio.v2 as imageio
import gymnasium as gym
import torch
from torch import nn
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack
from IPython.display import Video, display

# Avoid CPU thread oversubscription for these fairly small networks.
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.version.split()[0])
print("Gymnasium:", gym.__version__, "SB3:", sb3.__version__, "Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU is fine for setup checks. Select a GPU runtime before long training.")
assert gym.__version__ == "1.3.0", "Use the pinned Gymnasium version for these checks"


## 2. Mount Drive and protect the baseline

Use a **new experiment folder**. The default looks for `selected_400000.zip` first and then the original `best_validation.zip`. The loader checks that the saved counter is exactly 400k. Do not substitute the 500,526 checkpoint.

`RESUME_FROM=""` below means load a copy of this parent, not random initialization. If you already ran this experiment, set it to this experiment's `latest.zip` in section 4.

In [ ]:
import hashlib
import shutil
from stable_baselines3.common.logger import configure

USE_DRIVE = True
ARCHITECTURE = "basic_cnn"
EXPERIMENT_NAME = "steering_stability_01"
EXPECTED_BASE_TRANSITIONS = 400_000
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/max-verstappen")
else:
    OUTPUT_ROOT = Path.cwd() / "racing_outputs"
BASE_FOLDER = OUTPUT_ROOT / "basic_cnn" / "gsde_recovery_01"
BASE_CHECKPOINT = BASE_FOLDER / "selected_400000.zip"
if not BASE_CHECKPOINT.exists():
    BASE_CHECKPOINT = BASE_FOLDER / "best_validation.zip"
if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError("Set BASE_CHECKPOINT to the original selected 400k ZIP in your Drive")
RUN_DIR = OUTPUT_ROOT / ARCHITECTURE / EXPERIMENT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = RUN_DIR / "logs"
TRAINING_SEEDS = tuple(range(1000, 1100))
VALIDATION_SEEDS = tuple(range(10000, 10020))
TEST_SEEDS = tuple(range(20000, 20020))
PUBLIC_SEEDS = (202, 42, 203)
assert not set(TRAINING_SEEDS) & set(VALIDATION_SEEDS + TEST_SEEDS + PUBLIC_SEEDS)
AUDIT_SEEDS = VALIDATION_SEEDS[:3]
# Keep architecture/rollout settings compatible with the selected parent.
CONFIG = {
    "architecture": ARCHITECTURE, "run_seed": 7,
    "training_seeds": list(TRAINING_SEEDS),
    "image_size": 84, "frame_stack": 4, "action_repeat": 2,
    "n_envs": 2, "n_steps": 512, "batch_size": 128, "n_epochs": 5,
    "learning_rate": 1e-4, "gamma": 0.99, "gae_lambda": 0.95,
    "clip_range": 0.2, "ent_coef": 0.0, "vf_coef": 0.5,
    "use_sde": True, "sde_sample_freq": 4, "log_std_init": -2.0,
    "max_grad_norm": 0.5, "target_kl": 0.03, "features_dim": 256,
}
STEERING_CHANGE_COEF = 0.01  # Experimental; not an established optimum.
REWARD_SPEC = {
    "type": "steering_change_squared", "coefficient": STEERING_CHANGE_COEF,
    "frequency": "once per repeated-action decision", "first_decision_penalty": 0.0,
}
SETTINGS = {"config": CONFIG, "training_reward": REWARD_SPEC}
settings_path = RUN_DIR / "settings.json"
if settings_path.exists() and json.loads(settings_path.read_text()) != SETTINGS:
    raise ValueError("Settings changed: choose a new experiment folder")
settings_path.write_text(json.dumps(SETTINGS, indent=2))
print("Parent:", BASE_CHECKPOINT)
print("Separate outputs:", RUN_DIR)


## 3. Environment, steering cost and unchanged CNN

The training reward is:

`sum of original rewards over the repeated action − 0.01 × (steering − previous steering)²`.

It uses **bounded controls actually sent to the simulator**, once per policy decision. A switch from -1 to +1 costs 0.04; a change from 0.2 to 0.3 costs 0.0001. The first action after reset has no cost. Previous steering resets with the episode, so one race does not penalize the next.

This does not directly reward turning less: a constant steering command incurs no change cost. It may still discourage a necessary rapid correction. Validation environments receive **no added penalty**, and the deployed driver has no steering filter. Training reward logs and original evaluation rewards are different quantities.

The time limit stays at 3,000 **raw simulator steps**, inside action repeat. The CNN still receives four grayscale 84×84 frames.

In [ ]:
class TrackSampler(gym.Wrapper):
    """Choose tracks from one explicit pool; the policy never receives the seed."""
    def __init__(self, env, seeds, random_order=True, sampler_seed=7):
        super().__init__(env)
        self.seeds = tuple(int(seed) for seed in seeds)
        if not self.seeds:
            raise ValueError("The track pool cannot be empty")
        self.random_order = random_order
        self.rng = np.random.default_rng(sampler_seed)
        self.index = 0
        self.track_seed = None

    def reset(self, *, seed=None, options=None):
        # Here seed controls the SAMPLER. The chosen track seed is passed on
        # to CarRacing. SB3 seeds each worker once before its first reset.
        if seed is not None:
            self.rng = np.random.default_rng(seed)
            self.index = 0
        if self.random_order:
            self.track_seed = int(self.rng.choice(self.seeds))
        else:
            self.track_seed = self.seeds[self.index % len(self.seeds)]
            self.index += 1
        observation, info = self.env.reset(seed=self.track_seed, options=options)
        return observation, {**info, "track_seed": self.track_seed}

    def step(self, action):
        observation, reward, terminated, truncated, info = self.env.step(action)
        return observation, reward, terminated, truncated, {
            **info, "track_seed": self.track_seed,
        }


class ActionRepeat(gym.Wrapper):
    """Repeat a decision, sum rewards, and count actual simulator steps."""
    def __init__(self, env, repeat=2):
        super().__init__(env)
        if repeat < 1:
            raise ValueError("repeat must be positive")
        self.repeat = repeat
        self.raw_steps = 0

    def reset(self, **kwargs):
        self.raw_steps = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        total_reward = 0.0
        for _ in range(self.repeat):
            observation, reward, terminated, truncated, info = self.env.step(action)
            self.raw_steps += 1
            total_reward += float(reward)
            # Never continue stepping after either kind of episode ending.
            if terminated or truncated:
                break
        return observation, total_reward, terminated, truncated, {
            **info, "raw_steps": self.raw_steps,
        }


class GrayResize(gym.ObservationWrapper):
    """RGB -> resized grayscale; keep uint8 until SB3 normalizes it once."""
    def __init__(self, env, size=84):
        super().__init__(env)
        self.size = size
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(size, size, 1), dtype=np.uint8,
        )

    def observation(self, observation):
        image = Image.fromarray(observation).convert("L")
        image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.uint8)[..., None].copy()


class SteeringChangePenalty(gym.Wrapper):
    """Training-only reward cost for changes in executed steering."""
    def __init__(self, env, coefficient):
        super().__init__(env)
        if not np.isfinite(coefficient) or coefficient < 0:
            raise ValueError("Coefficient must be finite and nonnegative")
        self.coefficient = float(coefficient)
        self.previous_steering = None

    def reset(self, **kwargs):
        self.previous_steering = None
        return self.env.reset(**kwargs)

    def step(self, action):
        steering = float(np.clip(action[0], -1.0, 1.0))
        delta = 0.0 if self.previous_steering is None else steering - self.previous_steering
        observation, reward, terminated, truncated, info = self.env.step(action)
        cost = self.coefficient * delta ** 2
        self.previous_steering = steering
        return observation, float(reward) - cost, terminated, truncated, {
            **info, "original_decision_reward": float(reward), "steering_cost": cost,
        }


def make_single_env(seeds, random_order, sampler_seed, render_mode=None, training=False):
    # TimeLimit is INSIDE action repetition: 3000 means raw simulator steps.
    env = gym.make(
        "CarRacing-v3", continuous=True, max_episode_steps=3000,
        render_mode=render_mode,
    )
    env = TrackSampler(env, seeds, random_order, sampler_seed)
    env = ActionRepeat(env, repeat=CONFIG["action_repeat"])
    if training:
        if tuple(seeds) != TRAINING_SEEDS or not random_order:
            raise ValueError("Training only supports the ordinary seed pool")
        env = SteeringChangePenalty(env, STEERING_CHANGE_COEF)
    env = GrayResize(env, size=CONFIG["image_size"])
    return Monitor(env)


def build_vec_env(seeds, n_envs=1, random_order=True, render_mode=None, training=False):
    factories = []
    for rank in range(n_envs):
        # Bind rank now, rather than letting every factory use the last rank.
        def factory(rank=rank):
            return make_single_env(
                seeds, random_order, CONFIG["run_seed"] + rank, render_mode, training,
            )
        factories.append(factory)
    # DummyVecEnv runs its workers sequentially on the CPU. It batches their
    # observations for the network; it does not make the physics run on a GPU.
    env = DummyVecEnv(factories)
    env = VecTransposeImage(env)  # (84,84,1) -> (1,84,84)
    env = VecFrameStack(env, n_stack=CONFIG["frame_stack"], channels_order="first")
    env.seed(CONFIG["run_seed"])
    return env


In [ ]:
class BasicCNN(BaseFeaturesExtractor):
    """Three convolutional layers followed by a 256-number feature vector."""
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        channels = observation_space.shape[0]
        self.convolutions = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, *observation_space.shape)
            flattened_size = self.convolutions(dummy).shape[1]
        self.projection = nn.Sequential(
            nn.Linear(flattened_size, features_dim), nn.ReLU(),
        )

    def forward(self, observations):
        # SB3 already converts uint8 pixels to float32 / 255 before this call.
        return self.projection(self.convolutions(observations))


Encoder = BasicCNN


## 4. Load independent parent and training models

The parent copy is never optimized. Only the training copy receives the new reward and optimizer learning rate. Adam state is retained, but partial rollouts and simulator state are not restored. A parent fingerprint and reward settings prevent accidental cross-experiment resumes.

In [ ]:
RESUME_FROM = ""  # Later: str(RUN_DIR / "latest.zip")

if "train_env" in globals():
    train_env.close()
base_model = PPO.load(str(BASE_CHECKPOINT), device=DEVICE)
BASE_STEPS = int(base_model.num_timesteps)
assert BASE_STEPS == EXPECTED_BASE_TRANSITIONS, f"Expected selected 400k; got {BASE_STEPS}"
base_config = getattr(base_model, "experiment_config", None)
if not isinstance(base_config, dict):
    raise ValueError("Missing parent configuration")
for key, value in CONFIG.items():
    if key != "learning_rate" and base_config.get(key) != value:
        raise ValueError(f"Incompatible parent setting: {key}")
PARENT = {
    "sha256": hashlib.sha256(BASE_CHECKPOINT.read_bytes()).hexdigest(),
    "transitions": BASE_STEPS,
}
parent_path = RUN_DIR / "parent.json"
if parent_path.exists() and json.loads(parent_path.read_text()) != PARENT:
    raise ValueError("Parent differs from this experiment")
parent_path.write_text(json.dumps(PARENT, indent=2))
protected_path = RUN_DIR / f"parent_{BASE_STEPS}.zip"
if protected_path.exists():
    assert hashlib.sha256(protected_path.read_bytes()).hexdigest() == PARENT["sha256"]
else:
    shutil.copy2(BASE_CHECKPOINT, protected_path)
if not RESUME_FROM and (RUN_DIR / "latest.zip").exists():
    raise FileExistsError("Set RESUME_FROM to this folder's latest.zip, or choose a new folder")
train_env = build_vec_env(TRAINING_SEEDS, n_envs=CONFIG["n_envs"], training=True)
model = PPO.load(RESUME_FROM or str(BASE_CHECKPOINT), env=train_env, device=DEVICE)
if RESUME_FROM:
    if (getattr(model, "experiment_config", None) != CONFIG or
        getattr(model, "training_reward", None) != REWARD_SPEC or
        getattr(model, "finetune_parent", None) != PARENT):
        raise ValueError("Resume does not match this experiment")
model.learning_rate = CONFIG["learning_rate"]
model._setup_lr_schedule()  # Refresh the loaded schedule, not just the config text.
model.experiment_config = CONFIG.copy()
model.training_reward = REWARD_SPEC.copy()
model.finetune_parent = PARENT.copy()
model.set_logger(configure(str(LOG_DIR / f"segment_{model.num_timesteps}_{time.time_ns()}"), ["stdout", "csv"]))
CONTEXT = {
    "python": sys.version.split()[0], "torch": str(torch.__version__),
    "gymnasium": gym.__version__, "sb3": sb3.__version__,
    "numpy": np.__version__, "pillow": metadata.version("Pillow"),
    "device": str(model.device), "threads": torch.get_num_threads(),
}
print("Baseline transitions:", BASE_STEPS)
print("Training transitions:", model.num_timesteps)
print("Device:", model.device)
print("Learning rate:", model.lr_schedule(1.0))


## 5. Measure the baseline before training

Run three validation roads with the **original reward**, then use exactly those roads for the candidate. They have already been used for selection: this is a quick comparison, not an untouched final test. Do not train on them. Public/final-test roads are not evaluated by this short notebook.

The diagnostic also measures mean squared changes in executed steering. Lower is smoother, but a motionless or slow car can also be smooth: lap completion and penalized time are the deciding measures.

In [ ]:
def evaluate_tracks(policy, seeds, split="validation", video_path=None):
    """Evaluate fixed tracks without updating weights; optionally stream video."""
    pools = {
        "validation": VALIDATION_SEEDS, "test": TEST_SEEDS,
        "public": PUBLIC_SEEDS, "training": TRAINING_SEEDS,
    }
    if split not in pools or not set(seeds).issubset(set(pools[split])):
        raise ValueError("Evaluation seeds must belong to the requested split")
    if video_path is not None and len(seeds) != 1:
        raise ValueError("Record one track at a time")
    rows = []
    for seed in seeds:
        env = build_vec_env(
            (seed,), n_envs=1, random_order=False,
            render_mode="rgb_array" if video_path else None,
        )
        writer = None
        previous_steering = None
        steering_deltas = []
        try:
            observation = env.reset()
            total_reward = 0.0
            if video_path:
                # Stream to disk rather than accumulating hundreds of MB in RAM.
                writer = imageio.get_writer(
                    str(video_path), fps=50 / CONFIG["action_repeat"],
                )
            while True:
                action, _ = policy.predict(observation, deterministic=True)
                steering = float(action[0, 0])
                if previous_steering is not None:
                    steering_deltas.append((steering - previous_steering) ** 2)
                previous_steering = steering
                observation, rewards, dones, infos = env.step(action)
                total_reward += float(rewards[0])
                if writer and not dones[0]:
                    frame = env.envs[0].render()
                    frame = Image.fromarray(frame).resize((480, 320))
                    writer.append_data(np.asarray(frame))
                if dones[0]:
                    info = infos[0]  # Metadata belongs to the episode that ended.
                    # SB3 auto-resets: observation is now the NEW episode's image.
                    # The final image, if needed, is info['terminal_observation'].
                    if not info.get("TimeLimit.truncated", False) and "lap_finished" not in info:
                        raise RuntimeError("This Gymnasium version lacks lap completion metadata")
                    finished = bool(info.get("lap_finished", False))
                    steps = int(info["raw_steps"])
                    lap_seconds = steps / 50.0 if finished else None
                    rows.append({
                        "split": split, "track_seed": int(seed),
                        "finished": finished, "raw_steps": steps,
                        "lap_seconds": lap_seconds,
                        "penalized_seconds": lap_seconds if finished else 70.0,
                        "reward": total_reward,
                        "mean_squared_steering_change": float(np.mean(steering_deltas)) if steering_deltas else 0.0,
                    })
                    break
        finally:
            if writer is not None:
                writer.close()
            env.close()
    return rows


def summarize(rows):
    finishes = [row["lap_seconds"] for row in rows if row["finished"]]
    return {
        "tracks": len(rows),
        "completion_rate": sum(row["finished"] for row in rows) / len(rows),
        "mean_penalized_seconds": float(np.mean([row["penalized_seconds"] for row in rows])),
        "mean_finishing_seconds": float(np.mean(finishes)) if finishes else None,
        "mean_reward": float(np.mean([row["reward"] for row in rows])),
    }


def save_checkpoint(policy, name):
    """Write a complete SB3 model, including optimizer state, to Drive/local disk."""
    final_path = RUN_DIR / f"{name}.zip"
    temporary_path = RUN_DIR / f".{name}.tmp.zip"
    policy.save(str(temporary_path))
    os.replace(temporary_path, final_path)
    return final_path


In [ ]:
BASELINE_ROWS = evaluate_tracks(base_model, AUDIT_SEEDS, split="validation")
BASELINE_SUMMARY = summarize(BASELINE_ROWS)
BASELINE_REPORT = {"parent": PARENT, "context": CONTEXT, "summary": BASELINE_SUMMARY, "tracks": BASELINE_ROWS}
(RUN_DIR / "baseline_audit.json").write_text(json.dumps(BASELINE_REPORT, indent=2))
print("BASELINE:", BASELINE_SUMMARY)
for row in BASELINE_ROWS:
    print(row)


## 6. Train a short segment on ordinary roads

Only enable this after the baseline audit. Default limit: **10k additional transitions from the parent**, capped at about **12 minutes per invocation**. PPO normally finishes full rollouts, so the transition target may overshoot. A wall-clock stop during collection discards the final incomplete rollout; earlier optimizer updates remain saved. The counter counts interactions, not optimizer updates.

The callback checks elapsed time during collection; an individual optimizer update, simulator call, or Drive save can overrun it. This is not a strict 30-minute end-to-end guarantee. If interrupted, the `finally` block attempts a save, but a runtime crash can prevent it. Periodic checkpoints reduce the loss.

Do not repeatedly run this cell expecting a new 10k each time: the target is cumulative, parent+10k. To resume after a restart, use section 4's `RESUME_FROM` and rerun the comparison baseline in the new runtime.

In [ ]:
class ShortRunCallback(BaseCallback):
    def __init__(self, seconds, save_every=5000):
        super().__init__()
        self.seconds = seconds
        self.save_every = save_every

    def _on_training_start(self):
        self.started = time.perf_counter()
        self.next_save = (self.model.num_timesteps // self.save_every + 1) * self.save_every

    def _on_step(self):
        if self.model.num_timesteps >= self.next_save:
            save_checkpoint(self.model, "latest")
            self.next_save += self.save_every
        if time.perf_counter() - self.started >= self.seconds:
            print("Wall-clock budget reached; saving previous completed updates.")
            return False
        return True

RUN_TRAINING = False  # Set True after section 5 finishes.
ADDITIONAL_TIMESTEPS = 10_000
TRAIN_SECONDS = 12 * 60
TARGET_TOTAL_TIMESTEPS = BASE_STEPS + ADDITIONAL_TIMESTEPS
if RUN_TRAINING:
    if "BASELINE_ROWS" not in globals() or BASELINE_REPORT["parent"] != PARENT:
        raise RuntimeError("Run the baseline audit in this session first")
    remaining = max(0, TARGET_TOTAL_TIMESTEPS - model.num_timesteps)
    if remaining:
        start_steps = model.num_timesteps
        started = time.perf_counter()
        try:
            model.learn(total_timesteps=remaining, reset_num_timesteps=False,
                        callback=ShortRunCallback(TRAIN_SECONDS), progress_bar=False)
        finally:
            path = save_checkpoint(model, "latest")
            print("Saved:", path)
            print("Total transitions:", model.num_timesteps)
            print("This segment:", model.num_timesteps - start_steps)
            print("Training minutes:", (time.perf_counter() - started) / 60)
    else:
        print("Cumulative target already reached; evaluate before extending it.")
else:
    print("Training disabled. Complete the baseline audit, then set RUN_TRAINING=True.")


## 7. Compare the candidate using the original game reward

Run after training. Completion must not decrease, and mean penalized time should improve. Three roads provide limited evidence, not a guarantee for the mystery track. A smoother steering diagnostic by itself does not mean a better driver. Keep the baseline if results are worse or uncertain.

In [ ]:
candidate_rows = evaluate_tracks(model, AUDIT_SEEDS, split="validation")
candidate_summary = summarize(candidate_rows)
print("BASELINE:", BASELINE_SUMMARY)
print("CANDIDATE:", candidate_summary)
print("Penalized-time change (negative is better):",
      candidate_summary["mean_penalized_seconds"] - BASELINE_SUMMARY["mean_penalized_seconds"])
for row in candidate_rows:
    print(row)
report = {
    "parent": PARENT, "context": CONTEXT, "training_reward": REWARD_SPEC,
    "candidate_transitions": int(model.num_timesteps),
    "baseline": BASELINE_SUMMARY, "candidate": candidate_summary,
    "baseline_tracks": BASELINE_ROWS, "candidate_tracks": candidate_rows,
}
(RUN_DIR / f"comparison_{model.num_timesteps}.json").write_text(json.dumps(report, indent=2))
if model.num_timesteps <= BASE_STEPS:
    print("No additional training transitions: this is not a trained candidate yet.")


## 8. Save candidate weights separately

The training cell already saves `latest.zip` including optimizer state. This optional PTH is an inference bundle with the same image/control interface as the baseline; no steering filter or training reward executes at evaluation. It is labelled as a candidate and does not replace your release. Only export after comparing it. The baseline remains available in its original folder and as the protected parent copy.

In [ ]:
EXPORT_CANDIDATE = False
if EXPORT_CANDIDATE:
    if model.num_timesteps <= BASE_STEPS:
        raise RuntimeError("The candidate has not collected extra training experience")
    candidate_checkpoint = save_checkpoint(model, f"candidate_{model.num_timesteps}")
    output = RUN_DIR / f"steering_candidate_{model.num_timesteps}_weights.pth"
    torch.save({
        "format": "carracing-policy-v1", "config": CONFIG,
        "num_timesteps": int(model.num_timesteps),
        "policy_state_dict": {k: v.detach().cpu() for k, v in model.policy.state_dict().items()},
        "training_reward": REWARD_SPEC, "finetune_parent": PARENT,
        "source_sha256": hashlib.sha256(candidate_checkpoint.read_bytes()).hexdigest(),
        "sb3_version": str(sb3.__version__), "gymnasium_version": gym.__version__,
    }, output)
    print("Candidate weights:", output)
else:
    print("No weights exported. Keep your released baseline unless comparison supports a change.")


## Sources and limits

- [Gymnasium CarRacing](https://gymnasium.farama.org/environments/box2d/car_racing/): original tile-progress reward and per-frame time cost.
- [PPO paper](https://arxiv.org/abs/1707.06347): the policy optimization algorithm remains PPO.

The steering penalty and its coefficient are an engineering hypothesis, not a result established by these sources. It changes the training objective and does not guarantee preservation of the original optimal policy. Short pipeline checks are not evidence of faster driving.